In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/processed/policies_clean.csv")
assert len(df) == 676_789, "Wrong table loaded"
print(df.shape)

RANDOM_STATE = 42
train, test = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["HasClaim"]
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

# sanity check: both parts should look alike
for name, d in [("train", train), ("test", test)]:
    print(name, len(d),
          "| claim-policy rate:", round(d["HasClaim"].mean(), 4),
          "| freq:", round(d["ClaimNb"].sum() / d["Exposure"].sum(), 4),
          "| mean exposure:", round(d["Exposure"].mean(), 3),
          "| missing-sev share (claimants):", round(d.loc[d["ClaimNb"] > 0, "MissingSev"].mean(), 3))

# freeze the split so it never changes between runs
train[["IDpol"]].to_csv("../data/processed/train_ids.csv", index=False)
test[["IDpol"]].to_csv("../data/processed/test_ids.csv", index=False)

(676789, 21)
train 541431 | claim-policy rate: 0.0503 | freq: 0.1009 | mean exposure: 0.527 | missing-sev share (claimants): 0.27
test 135358 | claim-policy rate: 0.0503 | freq: 0.101 | mean exposure: 0.529 | missing-sev share (claimants): 0.262


In [14]:
assert len(train) + len(test) == len(df)
assert set(train["IDpol"]).isdisjoint(test["IDpol"])
print("split checks passed")

split checks passed


In [15]:
feat = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Area", "Density", "Region"]
train_keys = set(train[feat].itertuples(index=False, name=None))
twin = [row in train_keys for row in test[feat].itertuples(index=False, name=None)]
print("share of test rows with an identical-feature row in train:", round(sum(twin) / len(twin), 3))

share of test rows with an identical-feature row in train: 0.326


In [16]:
import numpy as np

feat = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Area", "Density", "Region"]

key = df.groupby(feat).ngroup()                       # one id per identical-feature group
size = key.map(key.value_counts())
pairs = df[size == 2].assign(key=key[size == 2])      # groups of exactly 2 rows

p = pairs.groupby("key").agg(
    id_min=("IDpol", "min"),
    id_max=("IDpol", "max"),
    exp_sum=("Exposure", "sum"),
)
p["id_gap"] = p["id_max"] - p["id_min"]

# chance benchmark: random pairs of rows
rng = np.random.default_rng(0)
a, b = rng.choice(df["Exposure"].to_numpy(), size=(2, len(p)))
c, d = rng.choice(df["IDpol"].to_numpy(), size=(2, len(p)))

print("groups of exactly 2 rows:", len(p))
print("exposure sum <= 1 | twins:", round((p["exp_sum"] <= 1.005).mean(), 3),
      "| random pairs:", round(((a + b) <= 1.005).mean(), 3))
print("ID gap <= 10 | twins:", round((p["id_gap"] <= 10).mean(), 3),
      "| random pairs:", round((abs(c - d) <= 10).mean(), 4))
print(p["id_gap"].describe().round(0))

groups of exactly 2 rows: 87535
exposure sum <= 1 | twins: 0.96 | random pairs: 0.423
ID gap <= 10 | twins: 0.86 | random pairs: 0.0
count      87535.0
mean       40198.0
std       301544.0
min            1.0
25%            1.0
50%            1.0
75%            3.0
max      5100561.0
Name: id_gap, dtype: float64


In [17]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv("../data/processed/policies_clean.csv")
assert len(df) == 676_789

feat = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Area", "Density", "Region"]
group = df.groupby(feat, sort=False).ngroup()      # one id per feature profile
print("distinct profiles:", group.nunique(), "of", len(df), "rows")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, df["HasClaim"], groups=group))
train = df.iloc[train_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)

# checks
assert len(train) + len(test) == len(df)
assert set(train["IDpol"]).isdisjoint(test["IDpol"])
train_keys = set(train[feat].itertuples(index=False, name=None))
twin = [r in train_keys for r in test[feat].itertuples(index=False, name=None)]
print("twin share after grouped split:", round(sum(twin) / len(twin), 3))   # must be 0.0

for name, d in [("train", train), ("test", test)]:
    print(name, len(d),
          "| claim-policy rate:", round(d["HasClaim"].mean(), 4),
          "| freq:", round(d["ClaimNb"].sum() / d["Exposure"].sum(), 4),
          "| mean exposure:", round(d["Exposure"].mean(), 3),
          "| missing-sev share (claimants):", round(d.loc[d["ClaimNb"] > 0, "MissingSev"].mean(), 3))

train[["IDpol"]].to_csv("../data/processed/train_ids.csv", index=False)
test[["IDpol"]].to_csv("../data/processed/test_ids.csv", index=False)
pd.DataFrame({"IDpol": df["IDpol"], "Group": group}).to_csv("../data/processed/policy_groups.csv", index=False)

distinct profiles: 527703 of 676789 rows
twin share after grouped split: 0.0
train 541432 | claim-policy rate: 0.0503 | freq: 0.1009 | mean exposure: 0.528 | missing-sev share (claimants): 0.268
test 135357 | claim-policy rate: 0.0502 | freq: 0.1011 | mean exposure: 0.527 | missing-sev share (claimants): 0.269


In [18]:
print("largest group:", group.value_counts().max())

largest group: 22


In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.evaluation import make_folds, PROCESSED

folds = make_folds()
fold_path = PROCESSED / "train_folds.csv"
if fold_path.exists():
    pd.testing.assert_frame_equal(pd.read_csv(fold_path), folds)     # existing file must match the code
    print("train_folds.csv already exists and matches make_folds()")
else:
    folds.to_csv(fold_path, index=False)
    print("train_folds.csv saved")

train_folds.csv already exists and matches make_folds()
